# TTC Subway Delay Analysis
Monthly delay statistics — 2023, 2024, and 2025 YTD.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../src").resolve()))

import polars as pl
import altair as alt
from sixsight.transforms.ttc_subway_delay import monthly_stats

In [2]:
CSV_PATH = Path("../data/raw/ttc-subway-delay-data/ttc-subway-delay-data-since-2025.csv")

df = pl.read_csv(CSV_PATH, schema_overrides={"Date": pl.Date})
print(df.schema)
df.head(3)

Schema({'_id': Int64, 'Date': Date, 'Time': String, 'Day': String, 'Station': String, 'Code': String, 'Min Delay': Int64, 'Min Gap': Int64, 'Bound': String, 'Line': String, 'Vehicle': Int64})


_id,Date,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Line,Vehicle
i64,date,str,str,str,str,i64,i64,str,str,i64
1,2025-01-01,"""02:10""","""Wednesday""","""BATHURST STATION""","""MUSAN""",5,9,"""E""","""BD""",5227
2,2025-01-01,"""02:30""","""Wednesday""","""DUNDAS STATION""","""MUIRS""",0,0,"""None""","""YU""",0
3,2025-01-01,"""02:32""","""Wednesday""","""BROADVIEW STATION""","""PUMST""",0,0,"""E""","""BD""",0


In [3]:
df_2023 = pl.read_excel(Path("../data/raw/ttc-subway-delay-data/ttc-subway-delay-2023.xlsx"))
df_2024 = pl.read_excel(Path("../data/raw/ttc-subway-delay-data/ttc-subway-delay-2024.xlsx"))

In [4]:
monthly = monthly_stats(pl.concat([df_2023, df_2024, df.drop("_id")]))
monthly

year_month,month_label,delays,total_delay,max_delay,major_delays
i32,str,u32,i64,i64,u32
202301,"""Jan 2023""",1738,5768,179,39
202302,"""Feb 2023""",1904,5266,309,44
202303,"""Mar 2023""",2105,6359,202,44
202304,"""Apr 2023""",1885,6543,420,47
202305,"""May 2023""",1919,6124,195,52
…,…,…,…,…,…
202509,"""Sep 2025""",1811,4179,149,18
202510,"""Oct 2025""",2027,5150,76,41
202511,"""Nov 2025""",2208,6187,480,32


In [5]:
month_order = monthly["month_label"].to_list()

base = alt.Chart(monthly).encode(
    x=alt.X("month_label:N", sort=month_order, title="Month"),
)

chart_total = base.transform_calculate(
    total_delay_days="datum.total_delay / 1440"
).mark_bar(color="#4C72B0").encode(
    y=alt.Y("total_delay_days:Q", title="Total Delay (days)"),
    tooltip=[
        alt.Tooltip("month_label:N", title="Month"),
        alt.Tooltip("total_delay_days:Q", title="Total Delay (days)", format=".2f"),
    ],
).properties(title="Total Delay per Month", width=650, height=250)

chart_delays = base.mark_bar(color="#55A868").encode(
    y=alt.Y("delays:Q", title="Number of Delays"),
    tooltip=[
        alt.Tooltip("month_label:N", title="Month"),
        alt.Tooltip("delays:Q", title="Delays"),
    ],
).properties(title="Number of Delays per Month", width=650, height=250)

chart_major = base.mark_bar(color="#DD8452").encode(
    y=alt.Y("major_delays:Q", title="Major Delays (>= 20 min)"),
    tooltip=[
        alt.Tooltip("month_label:N", title="Month"),
        alt.Tooltip("major_delays:Q", title="Major Delays"),
    ],
).properties(title="Major Delays per Month (>= 20 min)", width=650, height=250)

alt.vconcat(chart_total, chart_delays, chart_major)

alt.VConcatChart(...)